# Laboratorio 5

## Integrantes

| Nombre                            | Carnet | Usuario Git |
| --------------------------------- | ------ | ----------- |
| Edwin Jose Gabriel De Leon Garcia | 22809  | EJGDLG      |
| Gustavo Adolfo Cruz Bardales      | 22779  | G2309       |
| Josué Emanuel Say Garcia          | 22801  | JosueSay    |
| Mathew Alexander Cordero Aquino   | 22982  | donmatthiuz |


## Repositorio

[Link al Repositorio](https://github.com/donmatthiuz/RL/tree/lab5)

## Caso

Una empresa de gestión de tráfico urbano está evaluando el uso de RL con aproximación de función para controlar semáforos en una intersección con flujo vehicular variable. El estado del sistema es continuo: incluye densidad de vehículos, velocidad promedio y tiempo de espera acumulado en cada carril. La empresa necesita entender si la aproximación lineal es suficiente para este dominio antes de invertir en una solución basada en redes neuronales profundas.

Su grupo ha sido contratado para implementar y comparar aproximación tabular contra aproximación lineal sobre el mismo entorno, analizar las limitaciones de cada enfoque, y producir un dictamen técnico que justifique o descarte el salto hacia Deep RL.

## Task 1

Respondan las siguientes preguntas con argumentación técnica rigurosa. No implementen nada:

### Inciso 1

Para el dominio de control de semáforos descrito, diseñen un vector de características $x(s) \in \mathbb{R}^d$ para aproximación lineal. Especifiquen cada componente $x_i(s)$, su rango de valores y la justificación de su inclusión. Argumenten si las características que eligieron son suficientes para capturar las relaciones relevantes entre el estado y el valor, o si existen interacciones entre características que la aproximación lineal no puede representar sin una extensión explícita del vector.

**Respuesta:**

La intersección tiene cuatro accesos con dos carriles cada uno, así que $\ell = 1,\dots,8$ recorre los carriles. Cada carril aporta sus tres mediciones y se agregan tres características globales:

$$
\mathbf{x}(s) = \big[\ \rho_1, v_1, w_1,\ \dots,\ \rho_8, v_8, w_8,\ \phi,\ \tau,\ 1\ \big]^{\top} \in \mathbb{R}^{27}
$$

Las tres mediciones se escalan a $[0,1]$ dividiendo entre su máximo, porque vienen en unidades distintas y sin escalar la de números más grandes domina la actualización de $\mathbf{w}$. La estimación es $\hat{V}(s,\mathbf{w}) = \mathbf{w}^{\top}\mathbf{x}(s)$, con gradiente $\nabla_{\mathbf{w}}\hat{V}(s,\mathbf{w}) = \mathbf{x}(s)$.

| Componente | Rango | Justificación |
| :--- | :--- | :--- |
| $\rho_\ell$ (densidad) | $[0,1]$ | Cuántos vehículos esperan en el carril. Es la característica principal: más vehículos retenidos en rojo, peor el valor. |
| $v_\ell$ (velocidad) | $[0,1]$ | Distingue un carril que va saliendo de uno detenido en cola. Su peso debería salir con signo contrario al de la densidad. |
| $w_\ell$ (espera) | $[0,1]$ | Cuánto llevan esperando, que la densidad sola no dice. Sin ella el agente favorece siempre al eje con más tráfico. |
| $\phi$ (fase) | $\{0,1\}$ | Cuál eje tiene el verde ($1$ = norte-sur). Sin ella las mismas colas son ambiguas. |
| $\tau$ (tiempo en fase) | $[0,1]$ | Cuánto lleva la fase actual. Cambiar cuesta tiempo, así que sirve para que el agente no cambie a cada rato. |
| $1$ (sesgo) | constante | Permite que la estimación no pase por cero. |

La aproximación lineal suma cada característica por separado con un peso fijo, y eso alcanza para lo básico: más densidad en rojo es peor valor, la espera penaliza cada vez más y la fase desplaza la estimación. No alcanza para las interacciones entre características, y aquí hay cuatro que importan:

- **Densidad alta con velocidad baja.** Lo grave es que ocurran juntas (carril trabado), y el modelo no distingue ese carril de dos carriles normales con los mismos valores.
- **Colas de los dos ejes.** Darle verde al norte-sur depende de su cola comparada con la del este-oeste, y sumarlas por separado no expresa esa comparación.
- **Saturación.** El costo de un vehículo más se dispara cuando la cola llena el carril, y ningún peso fijo produce ese umbral.
- **Fase con las colas.** El efecto de la densidad del norte debería cambiar de signo según $\phi$, pero $\phi$ entra sumando y solo corre la estimación.


Las cuatro interacciones son conocidas de antemano, así que conviene agregarlas al vector antes que saltar a una red neuronal:

| Característica añadida | Interacción que resuelve |
| :--- | :--- |
| $\rho_\ell \cdot (1 - v_\ell)$ por carril | Carril trabado |
| Indicador de $\rho_\ell$ sobre un umbral crítico | Saturación |
| Diferencia de presión entre norte-sur y este-oeste | Comparación entre ejes |
| Esa diferencia multiplicada por $\phi$ | Fase con las colas |

El vector llega a unas 45 características y sigue siendo lineal en $\mathbf{w}$, así que conserva la convergencia garantizada y la interpretabilidad. La aproximación lineal sirve como primera solución aquí si el vector incluye esas interacciones, y su desempeño es la línea base para comparar contra Deep RL.

### Inciso 2

Dado el vector de características que diseñaron, ¿cuántos parámetros tiene su aproximador lineal? Compárenlo con el tamaño de la tabla $Q$ que necesitaría un método tabular si discretizaran cada variable de estado en $10$ niveles. ¿Qué consecuencia tiene esa diferencia sobre la capacidad de generalización y sobre el riesgo de sobreajuste?

**Respuesta**

Supuestos:

- Se cuenta $\hat{Q}$, no $\hat{V}$. Como la comparación es contra una tabla $Q$, el aproximador también debe estimar valor-acción, replicando el vector por acción.
- El sesgo no se discretiza. La constante $1$ es un término del modelo lineal, no una variable del estado, así que no aporta factor.
- La fase aporta factor $2$, no $10$. $\phi$ ya es binaria y solo tiene dos valores posibles; forzarla a 10 niveles dejaría 8 niveles que nunca se visitan.


Con 27 características y 2 acciones el aproximador tiene $27 \times 2 = 54$ parámetros, o unos $90$ con el vector extendido del inciso anterior.

La tabla $Q$ necesita una entrada por par estado-acción. Son 25 variables continuas a 10 niveles (24 mediciones de carril más $\tau$), por la fase binaria y por las 2 acciones:

$$
|\mathcal{S}| \times |\mathcal{A}| = 10^{25} \times 2 \times 2 = 4 \times 10^{25}
$$

| Enfoque | Parámetros |
| :--- | :--- |
| Aproximador lineal | $54$ |
| Tabla $Q$ discretizada | $4 \times 10^{25}$ |

La diferencia es de unos 24 órdenes de magnitud. La tabla tiene más entradas que las visitas posibles en cualquier entrenamiento imaginable y no cabe en memoria, así que la comparación no es entre dos opciones viables.

La tabla no generaliza nada: cada entrada se aprende por separado, así que un estado nunca visitado no tiene valor estimado aunque su vecino sí. Con $10^{24}$ celdas y cualquier presupuesto realista de episodios, casi todas quedan sin visitar y la política es esencialmente aleatoria fuera del puñado de celdas vistas.

Los 54 pesos hacen lo contrario: cada actualización desde un estado mueve la estimación de todos los estados con características parecidas. Por eso el aproximador da un valor razonable para configuraciones de tráfico que nunca ocurrieron, que es justamente lo que se necesita en un espacio continuo.

Con 54 parámetros el riesgo de sobreajuste es bajo, y el problema real es el opuesto: **subajuste**, porque el modelo no puede representar las interacciones del inciso 1 por más datos que reciba. La tabla tampoco llega a sobreajustar en el sentido clásico: con una o dos visitas por celda, la mayoría se queda en su valor de inicialización y el problema es falta de cobertura antes que memorización de ruido.

La discretización agrega además un error propio: cortar la densidad en 10 niveles hace que estados distintos caigan en la misma celda y reciban el mismo valor. Ese error es de diseño y no baja con más episodios, solo con una malla más fina que multiplica todavía más el tamaño de la tabla.

### Inciso 3

Argumenten formalmente por qué semi-gradient TD con aproximación lineal no converge al mínimo global de $J(w)$, sino al punto de TD. ¿Bajo qué condición sobre $\gamma$ el error en el punto de TD es más grave? ¿Cómo afecta esto la elección de $\gamma$ para su dominio específico?

### Inciso 4

Identifiquen cuál de los tres componentes de la tríada mortal es más relevante para el dominio de control de semáforos y justifiquen por qué. ¿Sería este un dominio donde el salto a Deep RL está justificado teóricamente, o la aproximación lineal debería ser suficiente?